# Fill UNK Words with LLM-Generated Phonemes

Extract all words still appearing as `UNK` in `labels.csv`, then use Azure OpenAI to generate RoLEX-format phonetic entries for each.

In [ ]:
import pandas as pd
import random
import time
from collections import Counter

# ── Load labels and extract UNK words ──
labels = pd.read_csv('../labels.csv')

# Replicate the lookup logic from phonemes.ipynb to know which words
# are genuinely missing (not just casing/hyphen variants)
rolex = pd.read_csv(
    'rolex.v1.txt',
    sep='\t',
    header=None,
    names=[
        'word_form',
        'lemma',
        'MSD_tag',
        'syllabification',
        'lexical_stress',
        'phonetic_transcription',
    ],
)
word_form_index = (
    rolex.drop_duplicates('word_form', keep='last')
    .set_index('word_form')['phonetic_transcription']
    .to_dict()
)

extra = pd.read_csv(
    'frequent_missing.tsv',
    sep='\t',
    header=None,
    names=[
        'word_form',
        'lemma',
        'MSD_tag',
        'syllabification',
        'lexical_stress',
        'phonetic_transcription',
    ],
)
word_form_index.update(
    extra.drop_duplicates('word_form')
    .set_index('word_form')['phonetic_transcription']
    .to_dict()
)


def _in_index(tok):
    if tok in word_form_index:
        return True
    if tok.lower() in word_form_index:
        return True
    cap = tok[0].upper() + tok[1:] if tok else tok
    if cap in word_form_index:
        return True
    return False


# Collect unique UNK words from labels.csv
unk_words_raw = set()
for phonemes, transcript in zip(labels['phonemes'], labels['transcript']):
    words = str(transcript).split()
    phon_parts = phonemes.split(' | ')
    for w, p in zip(words, phon_parts):
        if p == 'UNK':
            unk_words_raw.add(w)

# For hyphenated words, only generate entries for the missing parts
unk_words = set()
for w in unk_words_raw:
    if '-' in w:
        for part in w.split('-'):
            if part and not _in_index(part):
                unk_words.add(part)
    else:
        unk_words.add(w)

unk_list = sorted(unk_words)
print(f'Unique UNK words to generate: {len(unk_list)}')
print(
    f'  (from {len(unk_words_raw)} raw UNK tokens, {len(unk_words_raw) - len(unk_words)} resolved via hyphen split)'
)
print(f'Sample: {unk_list[:15]}')


Unique UNK words to generate: 0
  (from 0 raw UNK tokens, 0 resolved via hyphen split)
Sample: []


In [ ]:
# ── Sample 100 RoLEX lines as few-shot examples ──
random.seed(42)
sample_idx = random.sample(range(len(rolex)), 100)
examples = rolex.iloc[sample_idx]

example_lines = []
for _, row in examples.iterrows():
    example_lines.append('\t'.join(str(row[c]) for c in rolex.columns))

EXAMPLES_TEXT = '\n'.join(example_lines)

VALID_PHONEMES = {
    '.',
    '@',
    '1',
    'S',
    'Z',
    'a',
    'b',
    'c',
    'd',
    'e',
    'e_X',
    'f',
    'g',
    'gZ',
    'g_j',
    'gz',
    'h',
    'i',
    'i_0',
    'j',
    'je',
    'k',
    'k_j',
    'ks',
    'l',
    'm',
    'n',
    'o',
    'o_X',
    'p',
    'r',
    's',
    't',
    'tS',
    'ts',
    'u',
    'v',
    'w',
    'z',
}

print(f'Sampled {len(example_lines)} RoLEX examples')
print(f'Valid phonemes: {len(VALID_PHONEMES)}')
print(f'Example text length: {len(EXAMPLES_TEXT):,} chars')
print(f'\nFirst 3 examples:')
for line in example_lines[:3]:
    print(f'  {line}')


Sampled 100 RoLEX examples
Valid phonemes: 39
Example text length: 6,682 chars

First 3 examples:
  capilarul	capilar	Afpmsry	ca.pi.la.rul	capil'arul	k a p i l a r u l
  ntristații	întristat	Afpmpryy	n.tris.ta.ții	ntrist'ații	n t r i s t a ts i j
  hrincuțe	hrincuță	Ncfson	hrin.cu.țe	hrinc'uțe	h r i n k u ts e


In [ ]:
# ── Azure OpenAI client ──
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint='',
    api_version='',
    api_key='',
)
DEPLOYMENT = 'gpt-5.4'

# ── Pricing (per 1M tokens) ──
PRICE_CACHED_INPUT = 0.25  # cached system prompt
PRICE_INPUT = 2.5  # non-cached input (user message)
PRICE_OUTPUT = 15.0

# ── System prompt: static content → cached by Azure ──
SYSTEM_PROMPT = f"""You are a Romanian linguistics expert producing entries for the RoLEX phonetic lexicon.

Each entry is ONE tab-separated line with exactly 6 fields:
word_form\tlemma\tMSD_tag\tsyllabification\tlexical_stress\tphonetic_transcription

The phonetic_transcription field uses SPACE-SEPARATED tokens. You must use ONLY the 39 symbols defined below. No other symbols are allowed.

═══════════════════════════════════════════════════════════
PHONEME INVENTORY (39 symbols) — based on X-SAMPA notation
═══════════════════════════════════════════════════════════

── VOWELS (7 monophthongs) ──

  a   = IPA /a/  — open front unrounded vowel
        Romanian letters: a
        Examples: "mare" → m a r e, "casă" → k a s @

  e   = IPA /e/  — close-mid front unrounded vowel
        Romanian letters: e (when syllabic/full)
        Examples: "merge" → m e r gZ e, "bere" → b e r e

  i   = IPA /i/  — close front unrounded vowel (FULL syllabic i)
        Romanian letters: i (when it forms its own syllable)
        Examples: "visa" → v i s a, "mic" → m i k

  o   = IPA /o/  — close-mid back rounded vowel
        Romanian letters: o
        Examples: "bord" → b o r d, "dor" → d o r

  u   = IPA /u/  — close back rounded vowel
        Romanian letters: u
        Examples: "murg" → m u r g, "urs" → u r s

  @   = IPA /ə/  — schwa (mid central vowel)
        Romanian letters: ă
        Examples: "casă" → k a s @, "băiat" → b @ i a t

  1   = IPA /ɨ/  — close central unrounded vowel
        Romanian letters: â, î
        Examples: "câmp" → k 1 m p, "în" → 1 n, "pâine" → p 1 i n e

── VOWEL MODIFIERS (diacritics applied to vowels) ──

  e_X = IPA /e̯/ — extra-short (non-syllabic) e
        X-SAMPA diacritic _X means "extra-short" (IPA breve  ̆).
        Used when "e" is a non-syllabic glide element in diphthongs
        like "ea" [e̯a] or "eo" [e̯o].
        Examples: "bea" → b e_X a, "seară" → s e_X a r @

  i_0 = IPA /i̥/ — voiceless (devoiced/whispered) i
        X-SAMPA diacritic _0 means "voiceless" (IPA ring-below  ̥).
        Used for the characteristic Romanian devoiced final -i that
        palatalizes the preceding consonant without a full vowel.
        Examples: "bani" → b a n i_0, "mulți" → m u l ts i_0

  o_X = IPA /o̯/ — extra-short (non-syllabic) o
        X-SAMPA diacritic _X means "extra-short" (IPA breve  ̆).
        Used when "o" is a non-syllabic glide in diphthongs like
        "oa" [o̯a].
        Examples: "soare" → s o_X a r e, "doar" → d o_X a r

── SEMIVOWELS / APPROXIMANTS (3) ──

  j   = IPA /j/  — palatal approximant (glide/semivowel)
        Romanian: the glide in diphthongs like "ai", "ei", "oi", "iu"
        Examples: "mai" → m a j, "fiu" → f i u (but "iarbă" → j a r b @)

  w   = IPA /w/  — labial-velar approximant (glide/semivowel)
        Romanian: the glide in diphthongs like "au", "ou"
        Examples: "sau" → s a w, "nou" → n o w

  je  = the diphthong /je/ as a single unit
        Used specifically when Romanian "e" is pronounced [je], such as
        the word "e" (meaning "is") → je, or at word-initial "e" in
        certain contexts where a /j/ on-glide appears.

── PLOSIVES (6) ──

  p   = IPA /p/  — voiceless bilabial plosive
        Examples: "pas" → p a s

  b   = IPA /b/  — voiced bilabial plosive
        Examples: "bun" → b u n

  t   = IPA /t/  — voiceless alveolar plosive
        Examples: "tos" → t o s

  d   = IPA /d/  — voiced alveolar plosive
        Examples: "dar" → d a r

  k   = IPA /k/  — voiceless velar plosive
        Romanian letters: c (before a, o, u, â, consonants), ch, k
        Examples: "cap" → k a p, "curs" → k u r s

  g   = IPA /g/  — voiced velar plosive
        Romanian letters: g (before a, o, u, â, consonants), gh
        Examples: "gust" → g u s t

── PALATALIZED PLOSIVES (2) ──

  k_j = IPA /kʲ/ — palatalized voiceless velar plosive
        X-SAMPA diacritic _j means "palatalized" (IPA  ʲ).
        Romanian letters: ch (before e, i), sometimes just c shifted
        toward the palatal region.
        Examples: "cheamă" → k_j e_X a m @, "chin" → k_j i n

  g_j = IPA /gʲ/ — palatalized voiced velar plosive
        X-SAMPA diacritic _j means "palatalized" (IPA  ʲ).
        Romanian letters: gh (before e, i)
        Examples: "ghid" → g_j i d, "ghemui" → g_j e m u j

── VOICELESS PALATAL PLOSIVE (1) ──

  c   = IPA /c/  — voiceless palatal plosive
        A front allophone of /k/ occurring in certain palatal contexts.
        Examples: "cioc" → tS o c, "măciucă" → m @ tS u c @

── FRICATIVES (6) ──

  f   = IPA /f/  — voiceless labiodental fricative
        Examples: "foc" → f o k

  v   = IPA /v/  — voiced labiodental fricative
        Examples: "viu" → v i w

  s   = IPA /s/  — voiceless alveolar fricative
        Examples: "sus" → s u s

  z   = IPA /z/  — voiced alveolar fricative
        Examples: "zar" → z a r

  S   = IPA /ʃ/  — voiceless postalveolar fricative
        Romanian letters: ș
        Examples: "șarpe" → S a r p e, "mașină" → m a S i n @

  Z   = IPA /ʒ/  — voiced postalveolar fricative
        Romanian letters: j
        Examples: "jar" → Z a r, "ージ" → (used for foreign /ʒ/ too)

── AFFRICATES (3) ──

  ts  = IPA /t͡s/ — voiceless alveolar affricate (a single unit, NOT t+s)
        Romanian letters: ț
        Examples: "țară" → ts a r @, "brăț" → b r @ ts

  tS  = IPA /t͡ʃ/ — voiceless postalveolar affricate (a single unit)
        Romanian letters: ce, ci, cea, cia, ciu, cio (c before front vowels)
        Examples: "cer" → tS e r, "cine" → tS i n e

  gZ  = /d͡ʒ/ — voiced postalveolar affricate (a single unit)
        Romanian letters: ge, gi, gea, gia (g before front vowels)
        This is the voiced counterpart of tS.
        Examples: "ger" → gZ e r, "geam" → gZ e_X a m

── CONSONANT CLUSTERS (treated as single tokens) (2) ──

  ks  = IPA /ks/ — voiceless velar plosive + voiceless alveolar fricative
        Romanian: the pronunciation of "x" in many words
        Examples: "Alex" → a l e ks, "excepție" → e ks tS e p ts i e

  gz  = IPA /gz/ — voiced velar plosive + voiced alveolar fricative
        Romanian: the pronunciation of "x" in voiced contexts
        Examples: "auxiliar" → a u gz i l i a r, "examen" → e gz a m e n

── NASALS (2) ──

  m   = IPA /m/  — bilabial nasal
        Examples: "mamă" → m a m @

  n   = IPA /n/  — alveolar nasal
        Examples: "nor" → n o r

── LIQUIDS (2) ──

  l   = IPA /l/  — alveolar lateral approximant
        Examples: "lac" → l a k

  r   = IPA /r/  — alveolar trill
        Examples: "rar" → r a r

── GLOTTAL (1) ──

  h   = IPA /h/  — voiceless glottal fricative
        Examples: "har" → h a r

── SYLLABLE BOUNDARY (1) ──

  .   = syllable break marker (IPA /./), used rarely in loanwords or
        special contexts to mark explicit syllable divisions within the
        phonetic transcription itself.

═══════════════════════════════════════════════════════════
RULES FOR PHONETIC TRANSCRIPTION
═══════════════════════════════════════════════════════════

1. Output ONLY space-separated phoneme tokens from the 39 symbols above.
2. Every token in the transcription must be one of the 39 listed symbols. Do NOT invent new symbols, modify existing ones, or use raw IPA.
3. Romanian diphthongs are transcribed component-by-component:
   - "ea" → e_X a  (non-syllabic e + a)
   - "oa" → o_X a  (non-syllabic o + a)
   - "ai" → a j    (a + palatal glide)
   - "au" → a w    (a + labial glide)
   - "iu" → i w    (i + labial glide, when u is a glide)
   - "ei" → e j    (e + palatal glide)
4. Word-final devoiced -i → use i_0, NOT i.
   Exception: if -i is a full syllable (e.g., monosyllabic "zi"), use i.
5. Romanian "ă" → @, "â"/"î" → 1, "ș" → S, "ț" → ts, "j" → Z.
6. Romanian "ce"/"ci" → tS + vowel; "ge"/"gi" → gZ + vowel.
7. Romanian "che"/"chi" → k_j + vowel; "ghe"/"ghi" → g_j + vowel.
8. Romanian "x" → ks (voiceless context) or gz (voiced context).
9. For foreign/borrowed words: transcribe based on the source-language pronunciation, approximated with the 39 symbols above. Do NOT romanianize the pronunciation of foreign words.

Output ONLY the tab-separated lines, one per word, no extra text or explanations.

Here are 100 example entries from the RoLEX lexicon:
{EXAMPLES_TEXT}"""

print('Client ready.')
print(f'System prompt: {len(SYSTEM_PROMPT):,} chars (static, cached)')


Client ready.
System prompt: 14,885 chars (static, cached)


In [ ]:
# ── Generate entries (parallel, thread-safe, incremental save + cost tracking) ──
import os, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

BATCH_SIZE = 100
SAVE_FILE = 'llm_generated_missing.tsv'
N_WORKERS = 16

# Resume support: load already-generated words
lock = threading.Lock()
done_words = set()
results = []
if os.path.exists(SAVE_FILE):
    with open(SAVE_FILE) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) == 6:
                done_words.add(parts[0])
                results.append(parts)
    print(f'Resuming: {len(done_words)} words already generated')

todo = [w for w in unk_list if w not in done_words]
print(f'Words to generate: {len(todo)}  (skipping {len(done_words)} already done)')

# Thread-safe cost tracking
total_cached_tokens = 0
total_input_tokens = 0
total_output_tokens = 0


def _cost():
    return (
        total_cached_tokens * PRICE_CACHED_INPUT
        + total_input_tokens * PRICE_INPUT
        + total_output_tokens * PRICE_OUTPUT
    ) / 1e6


def _save():
    """Write current results to disk (caller must hold lock)."""
    with open(SAVE_FILE, 'w') as f:
        for row in results:
            f.write('\t'.join(row) + '\n')


def generate_batch(words, max_retries=3):
    global total_cached_tokens, total_input_tokens, total_output_tokens
    user_msg = (
        'Generate one RoLEX entry per word (6 tab-separated fields, one line each):\n'
        + '\n'.join(words)
    )
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_msg},
                ],
                temperature=0,
            )
            usage = resp.usage
            if usage:
                cached = (
                    getattr(
                        getattr(usage, 'prompt_tokens_details', None),
                        'cached_tokens',
                        0,
                    )
                    or 0
                )
                with lock:
                    total_cached_tokens += cached
                    total_input_tokens += usage.prompt_tokens - cached
                    total_output_tokens += usage.completion_tokens
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)
                time.sleep(wait)
            else:
                print(f'  FAILED after {max_retries} retries: {e}')
                return None


def process_batch(batch_words):
    """Process one batch: call LLM, parse, return (batch_results, batch_failed)."""
    output = generate_batch(batch_words)
    if output is None:
        return [], list(batch_words)

    batch_results = []
    for line in output.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) != 6:
            line = line.strip('`').strip()
            parts = line.split('\t')
        if len(parts) == 6:
            phonemes = parts[5].split()
            invalid = [p for p in phonemes if p not in VALID_PHONEMES]
            if invalid:
                print(f"  Invalid phonemes for '{parts[0]}': {invalid}")
            batch_results.append(parts)
        else:
            print(f'  Malformed line ({len(parts)} fields): {line[:80]}')

    generated_words = {r[0] for r in batch_results}
    generated_lower = {r[0].lower() for r in batch_results}
    batch_failed = [
        w
        for w in batch_words
        if w not in generated_words and w.lower() not in generated_lower
    ]
    return batch_results, batch_failed


failed = []
batches = [todo[i : i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]

pbar = tqdm(total=len(batches), desc='Generating')
with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(process_batch, bw): bw for bw in batches}
    for future in as_completed(futures):
        batch_results, batch_failed = future.result()
        with lock:
            results.extend(batch_results)
            failed.extend(batch_failed)
            _save()
        pbar.set_postfix(cost=f'${_cost():.2f}', words=len(results), fail=len(failed))
        pbar.update(1)
pbar.close()

# Cost summary
print(f'\nDone! Generated: {len(results)} entries, Failed: {len(failed)} words')
print(f'\n── Cost ──')
print(
    f'  Cached input: {total_cached_tokens:>10,} tokens  ${total_cached_tokens * PRICE_CACHED_INPUT / 1e6:.4f}'
)
print(
    f'  Input:        {total_input_tokens:>10,} tokens  ${total_input_tokens * PRICE_INPUT / 1e6:.4f}'
)
print(
    f'  Output:       {total_output_tokens:>10,} tokens  ${total_output_tokens * PRICE_OUTPUT / 1e6:.4f}'
)
print(f'  Total:                              ${_cost():.2f}')
if failed:
    print(f'\nFailed words sample: {failed[:20]}')


Resuming: 11953 words already generated
Words to generate: 0  (skipping 11953 already done)


Generating: 0it [00:00, ?it/s]


Done! Generated: 11953 entries, Failed: 0 words

── Cost ──
  Cached input:          0 tokens  $0.0000
  Input:                 0 tokens  $0.0000
  Output:                0 tokens  $0.0000
  Total:                              $0.00


In [ ]:
# ── Retry failed words (parallel, smaller batches) ──
if failed:
    print(f'Retrying {len(failed)} failed words in batches of 10...')
    retry_batches = [failed[i : i + 10] for i in range(0, len(failed), 10)]
    still_failed = []

    pbar = tqdm(total=len(retry_batches), desc='Retrying')
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_batch, bw): bw for bw in retry_batches}
        for future in as_completed(futures):
            batch_results, batch_failed = future.result()
            with lock:
                results.extend(batch_results)
                still_failed.extend(batch_failed)
                _save()
            pbar.set_postfix(cost=f'${_cost():.2f}', words=len(results))
            pbar.update(1)
    pbar.close()

    print(
        f'After retry: {len(results)} total entries, {len(still_failed)} still failed'
    )
    print(f'Updated total cost: ${_cost():.2f}')
    if still_failed:
        print(f'Permanently failed: {still_failed}')
else:
    print('No failed words to retry.')

# ── Validate ──
gen_df = pd.read_csv(
    SAVE_FILE,
    sep='\t',
    header=None,
    names=[
        'word_form',
        'lemma',
        'MSD_tag',
        'syllabification',
        'lexical_stress',
        'phonetic_transcription',
    ],
)
print(f'\nFinal TSV: {len(gen_df)} entries')
print(f'Unique words: {gen_df["word_form"].nunique()}')
gen_df.head(10)


No failed words to retry.

Final TSV: 11953 entries
Unique words: 11953


,word_form,lemma,MSD_tag,syllabification,lexical_stress,phonetic_transcription
0,DII,=,Np,di.i.i,dii,d i i i
1,DIICOT,=,Np,di.i.cot,diicot,d i i k o t
2,DJ,=,Np,de.jei,dj,d e Z e j
3,DVD,=,Np,de.ve.de,dvd,d e v e d e
4,Dalai,=,Np,da.lai,dal'ai,d a l a j
5,Damme,=,Np,dam,me,d a m
6,Daur,=,Np,daur,daur,d a w r
7,Davies,=,Np,da.vies,davies,d e j v i z
8,Davis,=,Np,da.vis,davis,d e j v i s
9,Decca,=,Np,dec.ca,decca,d e k a
